# Notebook 03 — Regression Results
**Purpose:** Evaluate ERS cost predictor and run residual diagnostics

---
## What this notebook answers
1. How accurate is the cost model? (RMSE, MAE, R²)
2. Are residuals well-behaved? (systematic bias = bad)
3. Does the log transform help? (before vs. after comparison)
4. What features drive cost? (feature importances for Random Forest)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120

from src.config import get_config
from src.features.preprocessing import run_preprocessing

cfg = get_config()
X, df_hh = run_preprocessing()
print(f'Households: {len(df_hh)}')

## 1. Identify the Cost Target Column

In [ ]:
target_candidates = [c for c in df_hh.columns if 'Total Cost' in c]
print('Cost columns found:', target_candidates)

target_col = target_candidates[0] if target_candidates else None
if target_col:
    y = df_hh[target_col].fillna(0).values
    print(f'\nUsing: {target_col}')
    print(f'Shape: {y.shape}')
    print(f'Zero-cost households: {(y == 0).mean():.1%}')
    print(f'Non-zero stats:')
    print(pd.Series(y[y > 0]).describe().to_string())
else:
    print('No cost column found — run preprocessing first')

## 2. Train All Regressors

In [ ]:
from src.models.regressor import train_cost_regressor

if target_col:
    results = train_cost_regressor(X, y, cfg=cfg)

## 3. Regressor Comparison

In [ ]:
if 'results' in dir() and results:
    compare_df = pd.DataFrame(results).T

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # RMSE
    axes[0].bar(compare_df.index, compare_df['rmse'].astype(float),
                color='#2E4A8C', edgecolor='white')
    axes[0].set_title('RMSE by Model (lower = better)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('RMSE ($)')

    # R²
    colors_r2 = ['#2E4A8C' if v >= 0.25 else '#E84C3D'
                 for v in compare_df['r2'].astype(float)]
    axes[1].bar(compare_df.index, compare_df['r2'].astype(float),
                color=colors_r2, edgecolor='white')
    axes[1].axhline(y=0.25, color='#F39C12', linestyle='--', lw=1.8, label='0.25 target')
    axes[1].axhline(y=0.40, color='#E84C3D', linestyle=':', lw=1.5, label='0.40 leakage flag')
    axes[1].set_title('R² by Model (target > 0.25)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('R²')
    axes[1].legend(fontsize=8)

    for ax in axes:
        ax.spines[['top','right']].set_visible(False)

    plt.suptitle('ERS Cost Regressor Comparison',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(cfg.paths.report_dir + 'regression_comparison.png', bbox_inches='tight')
    plt.show()

## 4. Check Already-Saved Residual Plots

In [ ]:
import os
from IPython.display import Image, display

fig_dir = cfg.paths.report_dir
residual_plots = [f for f in os.listdir(fig_dir) if 'residuals' in f]

for plot in sorted(residual_plots):
    print(f'\n{plot}')
    display(Image(os.path.join(fig_dir, plot)))